# Aula 4 — População


## Leitura da tabela

A população de 2010 foi compatibilizada com os limites territoriais de 2022, conforme a nota do IBGE na planilha. O crescimento será calculado em habitantes: população de 2022 menos população de 2010.


In [1]:
import pandas as pd

arquivo = "CD2022_Populacao_2010_Compatibilizada_20231222.xlsx"
df = pd.read_excel(
    arquivo,
    sheet_name="Municípios",
    header=2,
    dtype={"COD. UF": str, "COD. MUNIC": str}
)


In [2]:
col_2010 = "População 2010 (Alterações de Limites até 2022)1"
col_2022 = "População Censo 2022"

df = df.dropna(subset=["COD. UF", "COD. MUNIC"]).copy()
df = df[["COD. UF", "UF", "COD. MUNIC", "NOME DO MUNICÍPIO", col_2010, col_2022]]

for coluna in [col_2010, col_2022]:
    df[coluna] = pd.to_numeric(df[coluna]).astype("int64")

df.head()


,COD. UF,UF,COD. MUNIC,NOME DO MUNICÍPIO,População 2010 (Alterações de Limites até 2022)1,População Censo 2022
0,11,RO,00015,Alta Floresta D'Oeste,24392,21494
1,11,RO,00023,Ariquemes,90353,96833
2,11,RO,00031,Cabixi,6313,5351
3,11,RO,00049,Cacoal,78574,86887
4,11,RO,00056,Cerejeiras,17029,15890


## População agregada por estado


In [3]:
pop_estado = (
    df.groupby(["COD. UF", "UF"], as_index=False)
      .agg(
          populacao_2010=(col_2010, "sum"),
          populacao_2022=(col_2022, "sum"),
      )
)


In [4]:
pop_estado["crescimento"] = pop_estado["populacao_2022"] - pop_estado["populacao_2010"]
pop_estado = pop_estado.sort_values("crescimento", ascending=False).reset_index(drop=True)

pop_estado


,COD. UF,UF,populacao_2010,populacao_2022,crescimento
0,35,SP,41262199,44411238,3149039
1,42,SC,6248436,7610361,1361925
2,52,GO,6001789,7056495,1054706
3,41,PR,10444526,11444380,999854
4,31,MG,19597330,20539989,942659
5,51,MT,3035122,3658649,623527
6,15,PA,7581051,8120131,539080
7,13,AM,3483985,3941613,457628
8,23,CE,8451644,8794957,343313
9,32,ES,3514952,3833712,318760


In [5]:
pop_estado.to_csv("populacao_por_estado.csv", index=False, encoding="utf-8-sig")
print(f"CSV por estado salvo com {len(pop_estado)} unidades da Federação.")


CSV por estado salvo com 27 unidades da Federação.


## População por município

A planilha já tem uma linha por município. A prévia mostra os 20 maiores crescimentos; o CSV terá todos os municípios.


In [6]:
pop_municipio = df.rename(columns={
    col_2010: "populacao_2010",
    col_2022: "populacao_2022"
}).copy()

pop_municipio["crescimento"] = (
    pop_municipio["populacao_2022"] - pop_municipio["populacao_2010"]
)
pop_municipio = pop_municipio.sort_values("crescimento", ascending=False).reset_index(drop=True)

pop_municipio.head(20)


,COD. UF,UF,COD. MUNIC,NOME DO MUNICÍPIO,populacao_2010,populacao_2022,crescimento
0,13,AM,02603,Manaus,1802014,2063689,261675
1,53,DF,00108,Brasília,2572159,2817381,245222
2,35,SP,50308,São Paulo,11253503,11451999,198496
3,35,SP,52205,Sorocaba,586816,723682,136866
4,52,GO,08707,Goiânia,1301912,1437366,135454
5,14,RR,00100,Boa Vista,284313,413486,129173
6,42,SC,05407,Florianópolis,421240,537211,115971
7,15,PA,05536,Parauapebas,153908,267836,113928
8,50,MS,02704,Campo Grande,786774,898100,111326
9,25,PB,07507,João Pessoa,723515,833932,110417


In [7]:
pop_municipio.to_csv("populacao_por_municipio.csv", index=False, encoding="utf-8-sig")
print(f"CSV por município salvo com {len(pop_municipio)} municípios.")


CSV por município salvo com 5570 municípios.
